# Phase 3: whisper-server Flutter integration validation (Pixel 10 Pro)

Loads the CSV produced by the validation run of the Flutter app
(`fase_flutter_results.csv`), computes quality metrics with the same functions as
Phases 1 and 2, and compares against the Phase 2 benchmark for `tiny-q5_0` to
confirm that the deployment pipeline preserves the benchmark results.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q jiwer

In [ ]:
import re
from pathlib import Path
import pandas as pd
from jiwer import wer as wer_fn, cer as cer_fn

# Paths (adjust)
FLUTTER_CSV = Path('/content/drive/MyDrive/experimento_whisper/fase_flutter_results.csv')
PHASE2_CSV  = Path('/content/drive/MyDrive/experimento_whisper/bench_fase2.csv')

OUT_DIR = Path('/content/fase_flutter_validation')
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert FLUTTER_CSV.exists(), f'missing {FLUTTER_CSV}'
assert PHASE2_CSV.exists(),  f'missing {PHASE2_CSV}'

## 2. Ground truth and metric functions (same as Phases 1 and 2)

In [ ]:
PHRASES_ALL = [
    ('utterance_01', 'initial_location',   'I am in classroom 0.3 building G'),
    ('utterance_02', 'initial_location',   'I am in classroom 2.1 on the second floor'),
    ('utterance_03', 'initial_location',   'I am at the main entrance of building G'),
    ('utterance_04', 'initial_location',   'I am in the AI laboratory on the ground floor of building G'),
    ('utterance_05', 'initial_location',   'I am next to the elevator on floor 1 of building G'),
    ('utterance_06', 'destination',        'I want to go to classroom 1.4'),
    ('utterance_07', 'destination',        'Take me to the gym in building CD'),
    ('utterance_08', 'destination',        'I want to go to the cafeteria'),
    ('utterance_09', 'destination',        'Take me to the nearest restroom on this floor'),
    ('utterance_10', 'destination',        'Could you guide me to the EPS faculty offices please'),
    ('utterance_11', 'navigation_command', 'I am in a hurry please use the stairs'),
    ('utterance_12', 'navigation_command', 'Repeat the instructions please'),
    ('utterance_13', 'navigation_command', 'I am lost I need help'),
    ('utterance_14', 'navigation_command', 'Cancel navigation and start again'),
    ('utterance_15', 'navigation_command', 'Stop and tell me where I am'),
    ('utterance_16', 'confirmation',       'Yes I am ready to continue'),
    ('utterance_17', 'confirmation',       'No that is not my location'),
    ('utterance_18', 'confirmation',       'I have arrived at the elevator on floor 2'),
    ('utterance_19', 'confirmation',       'Done I have reached my destination'),
    ('utterance_20', 'confirmation',       'Yes I can see the elevator and the doors are open'),
    ('utterance_21', 'hard_domain',        'I am in classroom 1.1 and I want to go to the EPS faculty offices'),
    ('utterance_22', 'hard_domain',        'Take me from classroom 0.1 to the AI laboratory on the ground floor'),
    ('utterance_23', 'hard_domain',        'I just came out of the elevator on the first floor of building G'),
    ('utterance_24', 'hard_domain',        'I do not know where I am I need help finding my location'),
    ('utterance_25', 'hard_domain',        'I am at the directory sign in building G next to the mentoring room'),
    ('utterance_26', 'edge_case',          'How do I get from building CD to building G'),
    ('utterance_27', 'edge_case',          'Where is the secretary office in building G'),
    ('utterance_28', 'edge_case',          'I need to find classroom 0.3 quickly'),
    ('utterance_29', 'edge_case',          'Could you tell me how to reach the AI laboratory from here'),
    ('utterance_30', 'edge_case',          'My destination is classroom 1.4 in building G please'),
]
GT_MAP = {row[0]: {'category': row[1], 'ground_truth': row[2]} for row in PHRASES_ALL}

DOMAIN_KEYWORDS = [
    '0.1','0.3','1.1','1.4','2.1',
    'building g','building cd',
    'elevator','stairs','cafeteria','ai laboratory','restroom',
    'faculty offices','main entrance','mentoring room',
    'secretary','directory sign','gym',
    'ground floor','first floor','second floor','floor 1','floor 2',
    'cancel','repeat','hurry','lost','stop','done','destination',
    'doors are open',
]

In [ ]:
_PUNCT_RE = re.compile(r"[,;:!?\"()\[\]]")
_TRAIL_DOT_RE = re.compile(r"(?<!\d)\.|\.(?!\d)")
_WS_RE = re.compile(r"\s+")
_SPOKEN = {'zero':'0','one':'1','two':'2','three':'3','four':'4',
           'five':'5','six':'6','seven':'7','eight':'8','nine':'9'}

def normalize_text(t):
    if not t: return ''
    t = str(t).lower().strip()
    t = _PUNCT_RE.sub(' ', t)
    t = _TRAIL_DOT_RE.sub('', t)
    return _WS_RE.sub(' ', t).strip()

def normalize_for_dka(t):
    t = normalize_text(t)
    pat = re.compile(r'\b(' + '|'.join(_SPOKEN) + r')\s+point\s+(' + '|'.join(_SPOKEN) + r')\b')
    return pat.sub(lambda m: f"{_SPOKEN[m.group(1)]}.{_SPOKEN[m.group(2)]}", t)

def safe_wer(gt, pred):
    if not gt: return 0.0
    if not pred: return 1.0
    return float(wer_fn(normalize_text(gt), normalize_text(pred)))

def safe_cer(gt, pred):
    if not gt: return 0.0
    if not pred: return 1.0
    return float(cer_fn(normalize_text(gt), normalize_text(pred)))

def sent_acc(pred, gt):
    return 1.0 if normalize_text(pred) == normalize_text(gt) else 0.0

def dka(pred, gt, kws):
    p = normalize_for_dka(pred); g = normalize_for_dka(gt)
    gks = [k for k in kws if k in g]
    if not gks: return None
    return sum(1 for k in gks if k in p) / len(gks)

## 3. Load Flutter validation CSV and compute per-row metrics

In [ ]:
flutter = pd.read_csv(FLUTTER_CSV)
flutter['transcription'] = flutter['transcription'].fillna('')
flutter['ground_truth']  = flutter['audio'].map(lambda a: GT_MAP[a]['ground_truth'])
flutter['category']      = flutter['audio'].map(lambda a: GT_MAP[a]['category'])
flutter['wer'] = flutter.apply(lambda r: safe_wer(r['ground_truth'], r['transcription']), axis=1)
flutter['cer'] = flutter.apply(lambda r: safe_cer(r['ground_truth'], r['transcription']), axis=1)
flutter['sentence_accuracy'] = flutter.apply(lambda r: sent_acc(r['transcription'], r['ground_truth']), axis=1)
flutter['dka'] = flutter.apply(lambda r: dka(r['transcription'], r['ground_truth'], DOMAIN_KEYWORDS), axis=1)

print(f'Flutter rows: {len(flutter)}  (expected: 3 runs x 30 audios = 90)')
flutter.head(3)

## 4. Load Phase 2 CSV, filter to tiny-q5_0, same metrics

In [ ]:
phase2 = pd.read_csv(PHASE2_CSV)
phase2 = phase2[phase2['model'] == 'tiny-q5_0'].copy()
phase2['transcription'] = phase2['transcription'].fillna('')
phase2['ground_truth']  = phase2['audio'].map(lambda a: GT_MAP[a]['ground_truth'])
phase2['wer'] = phase2.apply(lambda r: safe_wer(r['ground_truth'], r['transcription']), axis=1)
phase2['dka'] = phase2.apply(lambda r: dka(r['transcription'], r['ground_truth'], DOMAIN_KEYWORDS), axis=1)
phase2['sentence_accuracy'] = phase2.apply(lambda r: sent_acc(r['transcription'], r['ground_truth']), axis=1)
print(f'Phase 2 tiny-q5_0 rows: {len(phase2)}  (expected: 3 runs x 30 audios = 90)')

## 5. Cross-context summary: Phase 2 (on-device CLI) vs Flutter (on-device server)

In [ ]:
def summarize(df, label):
    dka_vals = df['dka'].dropna()
    return {
        'context': label,
        'n_rows': len(df),
        'wer_pct': round(df['wer'].mean() * 100, 2),
        'sentence_accuracy_pct': round(df['sentence_accuracy'].mean() * 100, 2),
        'dka_pct': round(dka_vals.mean() * 100, 2) if len(dka_vals) else None,
        'latency_ms_mean': round(df['latency_ms'].mean(), 1),
        'latency_ms_std':  round(df['latency_ms'].std(),  1),
    }

comparison = pd.DataFrame([
    summarize(phase2,  'Phase 2: whisper-cli per-call (Pixel 10 Pro)'),
    summarize(flutter, 'Flutter: whisper-server bundled (Pixel 10 Pro)'),
])
comparison.to_csv(OUT_DIR / 'fase_flutter_vs_cli.csv', index=False)
print(comparison.to_string(index=False))

## 6. Per-audio byte-exact transcription comparison

Both contexts run `tiny-q5_0` with the same prompt, greedy decoding and temperature
0, so the transcriptions should match byte for byte, phrase by phrase.

In [ ]:
# Pick run 1 from each (decoding is deterministic; runs 2 and 3 should be identical)
fl_r1 = flutter[flutter['run'] == 1][['audio', 'transcription']].rename(columns={'transcription': 'flutter_text'})
f2_r1 = phase2[phase2['run'] == 1][['audio', 'transcription']].rename(columns={'transcription': 'cli_text'})

merged = fl_r1.merge(f2_r1, on='audio', how='outer').sort_values('audio').reset_index(drop=True)
merged['exact_match']      = merged['flutter_text'] == merged['cli_text']
merged['normalized_match'] = merged.apply(
    lambda r: normalize_text(r['flutter_text']) == normalize_text(r['cli_text']), axis=1)

print(f'Audios: {len(merged)}')
print(f'Byte-exact matches:    {merged["exact_match"].sum()} / {len(merged)}')
print(f'Normalized matches:    {merged["normalized_match"].sum()} / {len(merged)}')
print()
mismatches = merged[~merged['normalized_match']]
if len(mismatches):
    print('Mismatches:')
    for _, r in mismatches.iterrows():
        print(f'  {r["audio"]}')
        print(f'    CLI:     {r["cli_text"]}')
        print(f'    Flutter: {r["flutter_text"]}')
else:
    print('No mismatches: Flutter integration reproduces the CLI benchmark exactly after normalization.')

merged.to_csv(OUT_DIR / 'transcription_match.csv', index=False)

## 7. Package outputs

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/fase_flutter_validation.zip'
shutil.make_archive(zip_path.replace('.zip',''), 'zip', str(OUT_DIR))
print(f'Saved {zip_path}')
print('Contents:')
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file(): print(f'  {f.relative_to(OUT_DIR)}')
files.download(zip_path)